In [1]:
# calculate month-wise paired spatial correlation of produced monthly rasters
# helps understand whether model predicts beyond normal climatology to fine scale variations

In [2]:
import os
import glob
import pandas as pd 
import numpy as np  
import rasterio
import xarray as xr 
import rioxarray as rxr
from pathlib import Path 
import scipy.stats as stats
from rasterio.enums import Resampling

#configurations
start_time = pd.Timestamp("2001-01-01")
end_time   = pd.Timestamp("2005-12-31")

monthly_rasters_dir = r"..\Data\Processed\PredictedRasters\Final_rasters"
template_1km_wgs = rxr.open_rasterio(r"..\Data\Datasets\TemplateRasters\template_raster_WGS_1km.tif")

In [4]:
# helpers
 
def resample_to_template(raster: xr.DataArray) -> xr.DataArray:
    """Reproject and clip raster to match the 1km WGS84 template."""
    return raster.rio.reproject_match(template, resampling=Resampling.bilinear)
 
 
def parse_tif_catalog(directory: str) -> pd.DataFrame:
    """
    Build a DataFrame of (filepath, date) for all .tif files in a directory.
    Assumes filenames end with YYYYMMDD.tif (last 8 chars before extension).
    Returns sorted by date.
    """
    files = sorted(glob.glob(os.path.join(directory, "*.tif")))
    if not files:
        raise FileNotFoundError(f"No .tif files found in {directory}")
    rows = []
    for f in files:
        stem = Path(f).stem          # filename without extension
        year_str = stem[-7:-3]         # last 8 chars = YYYYMMDD
        month_str = stem[-2:]
        day_int = 1
        try:
            date = pd.Timestamp(year= int(year_str), month=int(month_str), day=day_int)
            rows.append({"path": f, "date": date})
        except ValueError:
            continue
    return pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
 
 
def filter_period(catalog: pd.DataFrame, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    return catalog[(catalog["date"] >= start) & (catalog["date"] <= end)].reset_index(drop=True)
 
 
 
def save_raster(data: xr.DataArray, out_path: str, crs, nodata: float = -9999.0):
    """Write a 2D DataArray to GeoTiff, preserving CRS and nodata."""
    data = data.rio.write_crs(crs)
    data = data.rio.write_nodata(nodata)
    data.rio.to_raster(out_path)
 
def month_label(year: int, month: int) -> str:
    return f"{year}{month:02d}"

def calculate_correlation_of_two_rasters(
    raster1: xr.DataArray,
    raster2: xr.DataArray,
    align: bool = True,
    p_threshold: float = 0.05,
    min_valid: int = 3,
    dim: str = "band"):
 
    if align:
        raster2 = raster2.rio.reproject_match(raster1, resampling=Resampling.bilinear)
 
    da1 = raster1.squeeze(dim, drop=True) if dim in raster1.dims else raster1
    da2 = raster2.squeeze(dim, drop=True) if dim in raster2.dims else raster2
    v1 = da1.values.ravel().astype(float)
    v2 = da2.values.ravel().astype(float)
    valid_mask = ~(np.isnan(v1) | np.isnan(v2))
    if valid_mask.sum() < min_valid:
        raise ValueError(
            f"Only {valid_mask.sum()} jointly valid pixels found. "
            f"Need at least {min_valid} to compute spatial correlation."
        )
 
    # Single Spearman r and p across all valid pixel pairs
    r_val, p_val = stats.spearmanr(v1[valid_mask], v2[valid_mask])
 
    if p_val < p_threshold:
        # Write r at every jointly valid pixel (all pixels carry same scalar)
        both_valid = (~np.isnan(da1)) & (~np.isnan(da2))
        corr_raster = xr.where(both_valid, float(r_val), np.nan)
        # Preserve spatial coordinates from da1
        corr_raster = corr_raster.assign_coords(da1.coords)
    else:
        # Overall correlation not significant — return all-NaN raster
        corr_raster = xr.full_like(da1, fill_value=np.nan)
 
    if raster1.rio.crs is not None:
        corr_raster = corr_raster.rio.write_crs(raster1.rio.crs)
        corr_raster = corr_raster.rio.write_transform(raster1.rio.transform())

    spatial_mean = float(corr_raster.mean(skipna=True))
    return corr_raster, spatial_mean
 
# Build list of (year, month) to process
month_range = pd.period_range(start=start_time, end=end_time, freq="M")

In [5]:
catalog = filter_period(parse_tif_catalog(monthly_rasters_dir), start_time, end_time).reset_index(drop = True)
crs = rxr.open_rasterio(catalog["path"].iloc[0]).rio.crs

correlation_results = []

for period in month_range:
    year, month = period.year, period.month
    
    for year_x in range(year, end_time.year + 1):
        local_corr = pd.DataFrame()
        date1 = pd.Timestamp(year = year, month = month, day = 1)
        date2 = pd.Timestamp(year = year_x, month = month, day = 1)
        if year == year_x:
            continue
        else:
            raster1 = rxr.open_rasterio((catalog['path'].loc[catalog['date'] == pd.Timestamp(year = year_x, month = month, day = 1)]).item())
            raster2 = rxr.open_rasterio((catalog['path'].loc[catalog['date'] == pd.Timestamp(year = year, month = month, day = 1)]).item())
            _, corr_val = calculate_correlation_of_two_rasters(raster1, raster2, align=True)
            
            correlation_results.append({
                'base_date': date1,
                'comparison_date': date2,
                'correlation': corr_val
            })
correlation_df = pd.DataFrame(correlation_results)

In [6]:
correlation_df.correlation.mean()

np.float64(0.7533768699009431)

In [8]:
correlation_df.to_csv(r"..\Data\Processed\FinalValidation\interannual_month_correlation.csv", index = False)